In [ ]:
#imports
import pydicom as dcm
import os
import torch
import numpy as np
import pandas as pd
import cv2
from dataclasses import dataclass, field
from copy import deepcopy

dicom_dir_path = './dataset/PINN2'
phantom_tensors_dir =  './dataset/phantom_tensors'
patches_np_binary_dir = './dataset/patches_np_binary'
patches_plot_dir = './patches_plots'

os.makedirs(phantom_tensors_dir, exist_ok=True)
os.makedirs(patches_np_binary_dir, exist_ok=True)
os.makedirs(patches_plot_dir, exist_ok=True)

VESSEL_ROI_MARGIN = 3
PORTAL_VEIN_RADIUS = 0

In [136]:
# Data classes
@dataclass
class SliceInfo:
  image: np.ndarray = None
  vein_patch: np.ndarray = None
  z_mm: float = 0
  thickness_mm: float = 0
  kernel: str = ""
  cx: int = 0
  cy: int = 0
  radius: float = 0
  slope: float = 1
  intercept: float = 0
  pixel_size: float = 0
  patch_size: int = 0

@dataclass
class Metrics:
  contrast: list = field(default_factory=list)
  noise: list = field(default_factory=list)

@dataclass
class ScanSlices:
  vein_slices: list[SliceInfo] = field(default_factory=list)
  uid: int = 0
  acqui_num: int = 0
  metrics: Metrics = field(default_factory=Metrics)

In [ ]:
#Patch processing
def extract_main_circle(dicom_image: np.ndarray, phantom_threshhold: int, slice_info) -> tuple[int,int,float,np.ndarray]:
  image_norm = cv2.normalize(dicom_image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

  _, mask = cv2.threshold(image_norm, phantom_threshhold, 255, cv2.THRESH_BINARY)
  contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
  outter_circle = max(contours, key=cv2.contourArea)

  (cx, cy), phantom_radius = cv2.minEnclosingCircle(outter_circle)
  cx = int(cx)
  cy = int(cy)

  return cx,cy,phantom_radius

def find_vein(
      cx: int,
      cy: int,
      radius: float,
      radius_ratio: float,
      insert_angle: int, 
) -> tuple[int]:
  radius_to_vein = radius_ratio * radius

  theta = np.deg2rad(insert_angle)
  x = int(cx + radius_to_vein * np.cos(theta))
  y = int(cy + radius_to_vein * np.sin(theta))

  return (x, y)

def extract_patch(insert_center: tuple[int], image: np.ndarray, patch_size: int = 40) -> np.ndarray:
  half = patch_size // 2
  cx, cy = insert_center
  x1 = max(0, cx - half)
  x2 = min(image.shape[1], cx + half)
  y1 = max(0, cy - half)
  y2 = min(image.shape[0], cy + half)
  patch = {'image': image[y1:y2, x1:x2], 'cx': cx - x1, 'cy': cy - y1}

  return patch

In [ ]:
#Contrast-Noise Ratio
from scipy.ndimage import maximum_filter

def find_vessel_center(patch_hu: np.ndarray, cx: float, cy: float, search_radius: int = 4):
    """
    Find the single brightest local maximum within search_radius of (cx, cy).
    Returns the center of that peak.
    """
    height, width = patch_hu.shape
    Y, X = np.ogrid[:height, :width]
    search_mask = (X - cx)**2 + (Y - cy)**2 <= search_radius**2

    restricted = np.full_like(patch_hu, -np.inf)
    restricted[search_mask] = patch_hu[search_mask]

    local_max = maximum_filter(restricted, size=5)
    peak_mask = (restricted == local_max) & search_mask

    if not peak_mask.any():
        return cx, cy  # fallback to patch center

    peak_coords = np.argwhere(peak_mask)
    peak_vals = patch_hu[peak_mask]
    best_idx = np.argmax(peak_vals)
    peak_cy, peak_cx = peak_coords[best_idx]

    return float(peak_cx), float(peak_cy)


def cnr_mes_vein(patch_hu: np.ndarray, cx: float, cy: float, roi_radius: int = 4):
    """
    Compute CNR using a fixed-radius circle centered on the brightest
    local peak near (cx, cy).
    """
    roi_cx, roi_cy = find_vessel_center(patch_hu, cx, cy)

    height, width = patch_hu.shape
    Y, X = np.ogrid[:height, :width]
    target_mask = (X - roi_cx)**2 + (Y - roi_cy)**2 <= roi_radius**2

    target = patch_hu[target_mask]
    background = patch_hu[~target_mask]

    contrast = abs(np.mean(target) - np.mean(background))
    noise = np.sqrt(np.var(target) / target.size + np.var(background) / background.size)

    return contrast, noise, roi_cx, roi_cy


In [ ]:
def load_group_dcm_slices(directory: str) -> dict:
  #Mes. Vein range
  mes_vein_z_max_1mm = 1266.38
  mes_vein_z_min_1mm = 1253.18
  mes_vein_z_max_3mm = 1262.18
  mes_vein_z_min_3mm = 1254.18
  
  max_slices = 4

  scans = {}

  #Traverse trough all dcm files
  for file in os.listdir(directory):
    phantom_path = os.path.join(directory, file)
    ct_slice = dcm.dcmread(phantom_path)

    if not hasattr(ct_slice, "ImagePositionPatient") or ct_slice.SliceThickness == None:
      continue

    uid = ct_slice.SeriesInstanceUID
    z = float(ct_slice.ImagePositionPatient[2])

    if uid not in scans:
      scans[uid] = ScanSlices()
      scans[uid].uid = uid
      scans[uid].acqui_num = ct_slice.AcquisitionNumber

    slice_info = SliceInfo()
    slice_info.image = ct_slice.pixel_array.astype(np.float32)
    slice_info.z_mm = z
    slice_info.thickness_mm = float(ct_slice.SliceThickness)
    slice_info.kernel = str(ct_slice.ConvolutionKernel)
    slice_info.slope = ct_slice.RescaleSlope
    slice_info.intercept = ct_slice.RescaleIntercept
    slice_info.pixel_size = ct_slice.PixelSpacing[0]
      

    if len(scans[uid].vein_slices) < max_slices and ct_slice.SliceThickness == 1 and (mes_vein_z_min_1mm <= z <= mes_vein_z_max_1mm):
      slice_info.cx, slice_info.cy, slice_info.radius = extract_main_circle(slice_info.image, 50, slice_info)
      scans[uid].vein_slices.append(slice_info)

    if len(scans[uid].vein_slices) < max_slices and ct_slice.SliceThickness == 3 and (mes_vein_z_min_3mm <= z <= mes_vein_z_max_3mm):
      slice_info.cx, slice_info.cy, slice_info.radius = extract_main_circle(slice_info.image, 50, slice_info)
      scans[uid].vein_slices.append(slice_info)


  return list(scans.values())

In [ ]:
def dicom_to_patches(all_scans: list[ScanSlices], patch_size: int, patch_fn: callable) -> None:
  radius_ratio = 0.2
  insert_angle = 320
  #Traverse trough all dcm files
  new_scans = []
  for scan in all_scans:
    new_scan = deepcopy(scan)
    for ct_slice in new_scan.vein_slices:
      insert_center = find_vein(
        ct_slice.cx, 
        ct_slice.cy, 
        ct_slice.radius, 
        radius_ratio, 
        insert_angle,
      )
      patch = patch_fn(insert_center, ct_slice.image, patch_size)
      ct_slice.patch_size = patch_size
      ct_slice.vein_patch = patch
    new_scans.append(new_scan)

  return new_scans

def calculate_metrics(all_scans: list[ScanSlices]):
  for scan in all_scans:
    #CNR
    for vein_slice in scan.vein_slices:
      #Caluclate the values for the patches in HU 
      contrasts, noises = [], []
      contrast, noise, _, _ = cnr_mes_vein(
        vein_slice.vein_patch['image'] * vein_slice.slope + vein_slice.intercept, 
        vein_slice.vein_patch['cx'], 
        vein_slice.vein_patch['cy']
      )
      contrasts.append(contrast)
      noises.append(noise)

      scan.metrics.contrast.append(contrasts)
      scan.metrics.noise.append(noises)


In [ ]:
def create_tensors(all_scan_patches: list, filename: str, iq_csv_path: str) -> None:
  label_tensors = []
  insert_tensors = []
  settings_df = pd.read_csv(iq_csv_path)
  iq_tensors = []
  st_tensors = []
  acqui_num_tensor = []

  sorted_scans= sorted(all_scan_patches, key=lambda sp: sp.acqui_num)

  for scan in sorted_scans:
    #CNR patches to tensor
    slice_tensors = []
    for vein_slice in scan.vein_slices:
      slice_tensors.append(
        torch.stack([torch.from_numpy(vein_slice.vein_patch['image']).float()]).unsqueeze(1)
      )
    insert_tensors.append(torch.stack(slice_tensors))

    # metrics labels 
    contrast_per_insert = np.mean(scan.metrics.contrast, axis=0)
    noise_per_insert = np.mean(scan.metrics.noise, axis=0)
    
    metrics_per_insert = torch.tensor(
      [float(np.mean(scan.metrics.contrast)), float(np.mean(scan.metrics.noise))],
      dtype=torch.float32
    )

    label_tensors.append(metrics_per_insert)

    csv_condition = (settings_df['acqui_num'] == scan.acqui_num) \
                    & (settings_df['slice_thickness'] == scan.vein_slices[0].thickness_mm)
    # IQ level from CSV matched by acqui_num
    iq_row = settings_df[csv_condition]['iq_level'].values[0]
    iq_tensors.append(torch.tensor([iq_row], dtype=torch.float32))

    #Slice thickness from CSV
    slice_thick_row = settings_df[csv_condition]['slice_thickness'].values[0]
    st_tensors.append(torch.tensor([slice_thick_row], dtype=torch.float32))

    acqui_num_tensor.append(scan.acqui_num)

  torch.save(
    {
      'insert_patches': insert_tensors,
      'labels': torch.stack(label_tensors),
      'iq_level': torch.stack(iq_tensors),
      'slice_thickness': torch.stack(st_tensors),
      'acqui_num': torch.tensor(acqui_num_tensor)
    },
    os.path.join(phantom_tensors_dir,filename + '.pt')
  )

In [ ]:
exp = 'exp1_2'
patch_size = 40

all_scans = load_group_dcm_slices(dicom_dir_path)

new_scans = dicom_to_patches(all_scans, patch_size, extract_patch)
calculate_metrics(new_scans)
create_tensors(
  new_scans, 
  f'phantomx_rois_{patch_size}',
  './dataset/iq_level_per_scan_phantomx.csv'
)